# LeetCode 1614 — Maximum Nesting Depth of the Parentheses bằng Python

**Link đề:** https://leetcode.com/problems/maximum-nesting-depth-of-the-parentheses/
**Độ khó:** Easy · **Chủ đề:** String, Stack, Bracket Sequences

```python
class Solution:
    def maxDepth(self, s: str) -> int:
        ...
```

## Mục tiêu học
1. Hiểu **chuỗi ngoặc hợp lệ (VPS — Valid Parentheses String)** và **độ sâu lồng nhau** (nesting depth).
2. Đi từ cách trực giác `O(n²)` (đếm ngoặc trước mỗi ký tự) → **stack** → **biến đếm** `O(1)` bộ nhớ.
3. Hiểu invariant: tại mỗi vị trí, độ sâu = số `(` đã gặp − số `)` đã gặp.
4. Làm quen `itertools.accumulate` để tính **tổng tiền tố** của dãy `+1/-1`.
5. Trace từng ký tự, test đối chiếu, mở rộng: kiểm tra hợp lệ, nhiều loại ngoặc, tách chuỗi ngoặc (1111).

## 1. Đề bài

Cho chuỗi ngoặc **hợp lệ** `s` (gồm chữ số, `+ - * /`, `(` và `)`). Trả về **độ sâu lồng nhau** của `s` — số cặp ngoặc lồng nhau **lớn nhất**.

| `s` | Kết quả | Vì sao |
|---|:---:|---|
| `"(1+(2*3)+((8)/4))+1"` | `3` | chữ số `8` nằm trong 3 lớp ngoặc |
| `"(1)+((2))+(((3)))"` | `3` | chữ số `3` nằm trong 3 lớp ngoặc |
| `"()(())((()()))"` | `3` | cụm cuối có 3 lớp |

**Constraints:** `1 <= s.length <= 100`; `s` là VPS (đề đảm bảo).

Hiểu trực quan: đi từ trái sang phải, gặp `(` là **xuống một tầng**, gặp `)` là **lên một tầng**. Độ sâu lớn nhất là tầng thấp nhất từng xuống tới.

In [1]:
examples = [("(1+(2*3)+((8)/4))+1", 3), ("(1)+((2))+(((3)))", 3), ("()(())((()()))", 3)]
for s, expected in examples:
    print(f"{s!r:<26} -> kỳ vọng {expected}")

'(1+(2*3)+((8)/4))+1'      -> kỳ vọng 3
'(1)+((2))+(((3)))'        -> kỳ vọng 3
'()(())((()()))'           -> kỳ vọng 3


### 1.1. VPS là gì?

Một chuỗi là **VPS** nếu:
- là chuỗi rỗng hoặc một ký tự không phải ngoặc; hoặc
- là `AB` (nối hai VPS); hoặc
- là `(A)` với `A` là VPS.

Hệ quả thực tế: đếm từ trái sang phải, số `)` **không bao giờ vượt** số `(`, và cuối chuỗi hai số bằng nhau. Nhờ đề đảm bảo VPS, ta **không cần** kiểm tra lỗi.

## 2. Nền tảng Python

### 2.1. Duyệt từng ký tự và so sánh

`for ch in s` lấy lần lượt từng ký tự. So sánh `ch == "("`. Các ký tự khác (chữ số, phép toán) **không ảnh hưởng** độ sâu → bỏ qua.

In [2]:
s = "(1+(2))"
for i, ch in enumerate(s):
    kind = "mở" if ch == "(" else ("đóng" if ch == ")" else "khác")
    print(i, repr(ch), kind)

0 '(' mở
1 '1' khác
2 '+' khác
3 '(' mở
4 '2' khác
5 ')' đóng
6 ')' đóng


### 2.2. `str.count` trên một đoạn — đếm ngoặc trước vị trí `i`

`s[:i]` là đoạn trước vị trí `i`. `s[:i].count("(")` đếm số ngoặc mở trong đoạn đó.

In [3]:
s = "(1+(2*3)+((8)/4))+1"
i = s.index("8")          # vị trí đầu tiên của '8'
before = s[:i]
print("vị trí của 8:", i, "| đoạn trước:", repr(before))
print("số '(' trước:", before.count("("), "| số ')' trước:", before.count(")"))
print("độ sâu của '8':", before.count("(") - before.count(")"))

vị trí của 8: 11 | đoạn trước: '(1+(2*3)+(('
số '(' trước: 4 | số ')' trước: 1
độ sâu của '8': 3


### 2.3. Stack bằng list (nhắc lại)

`append` để đẩy vào, `pop` để lấy ra phần tử cuối. Độ dài stack = số ngoặc mở **chưa được đóng**.

In [4]:
stack = []
for ch in "(()":
    if ch == "(":
        stack.append(ch)
    elif ch == ")":
        stack.pop()
    print(repr(ch), "-> stack =", stack, "| độ sâu =", len(stack))

'(' -> stack = ['('] | độ sâu = 1
'(' -> stack = ['(', '('] | độ sâu = 2
')' -> stack = ['('] | độ sâu = 1


### 2.4. `max(a, b)` để cập nhật kỷ lục

Mẫu "giữ giá trị lớn nhất từng thấy": `best = max(best, current)`.

In [5]:
best = 0
for current in [1, 3, 2, 4, 0]:
    best = max(best, current)
    print("current =", current, "| best =", best)

current = 1 | best = 1
current = 3 | best = 3
current = 2 | best = 3
current = 4 | best = 4
current = 0 | best = 4


## 3. Cách 1 — Trực giác `O(n²)`: độ sâu của từng ký tự

Đúng như hint của LeetCode: *độ sâu của một ký tự = (số `(` trước nó) − (số `)` trước nó)*. Tính cho **mọi** vị trí rồi lấy max.

Ta có thể chỉ xét vị trí ngay **sau** một dấu `(` (đó là lúc độ sâu vừa tăng), nhưng để đơn giản cứ xét mọi vị trí `i` từ `0` đến `n` (đoạn `s[:i]`).

In [6]:
def max_depth_brute(s: str) -> int:
    best = 0
    for i in range(len(s) + 1):
        prefix = s[:i]
        depth = prefix.count("(") - prefix.count(")")
        best = max(best, depth)
    return best

for s, expected in examples:
    print(max_depth_brute(s), "| kỳ vọng", expected)

3 | kỳ vọng 3
3 | kỳ vọng 3
3 | kỳ vọng 3


## 4. Nút thắt của cách 1

Với mỗi `i`, ta đếm lại toàn bộ `s[:i]` — `O(i)` việc. Tổng `1 + 2 + ... + n ≈ n²/2`.

Nhưng `s[:i+1]` chỉ hơn `s[:i]` **đúng một ký tự**! Độ sâu tại `i+1` = độ sâu tại `i` + (1 nếu ký tự đó là `(`, −1 nếu `)`, 0 nếu khác). → Chỉ cần **một biến cộng dồn**.

In [7]:
s = "(1+(2*3)+((8)/4))+1"
n = len(s)
print("số phép đếm cách 1 ≈", sum(range(n + 2)), "| cách cộng dồn:", n)

số phép đếm cách 1 ≈ 210 | cách cộng dồn: 19


## 5. Cách 2 — Stack

Đẩy `(` vào stack, gặp `)` thì `pop`. Sau mỗi lần đẩy, `len(stack)` là độ sâu hiện tại → cập nhật max.

Cách này đúng và tự nhiên (giống kiểm tra ngoặc hợp lệ — bài 20), nhưng ta không dùng **nội dung** stack, chỉ dùng **độ dài**.

In [8]:
def max_depth_stack(s: str) -> int:
    stack = []
    best = 0
    for ch in s:
        if ch == "(":
            stack.append(ch)
            best = max(best, len(stack))
        elif ch == ")":
            stack.pop()
    return best

print([max_depth_stack(s) for s, _ in examples])

[3, 3, 3]


### 5.1. Insight — thay stack bằng một biến đếm

Stack chỉ chứa toàn ký tự `(` giống nhau → chỉ cần biết **có bao nhiêu** → biến `depth`:
- `(` → `depth += 1`, rồi `best = max(best, depth)`.
- `)` → `depth -= 1`.
- Ký tự khác → bỏ qua.

Chỉ cần cập nhật `best` **sau khi tăng** (sau `(`), vì độ sâu chỉ đạt đỉnh mới ngay sau một dấu mở.

## 6. Cách 3 (lời giải chính) — biến đếm `depth`

### 6.1. Từng đoạn

In [9]:
s = "(()(()))"
depth = best = 0           # gán cùng lúc hai biến bằng 0
for ch in s:
    if ch == "(":
        depth += 1
        best = max(best, depth)
    elif ch == ")":
        depth -= 1
print("depth cuối =", depth, "(VPS luôn về 0) | best =", best)

depth cuối = 0 (VPS luôn về 0) | best = 3


### 6.2. Ghép thành hàm

In [10]:
def max_depth_counter(s: str) -> int:
    depth = best = 0
    for ch in s:
        if ch == "(":
            depth += 1
            if depth > best:
                best = depth
        elif ch == ")":
            depth -= 1
    return best

for s, expected in examples:
    print(f"{s!r:<26} -> {max_depth_counter(s)} | kỳ vọng {expected}")

'(1+(2*3)+((8)/4))+1'      -> 3 | kỳ vọng 3
'(1)+((2))+(((3)))'        -> 3 | kỳ vọng 3
'()(())((()()))'           -> 3 | kỳ vọng 3


## 7. Cách 4 — Tổng tiền tố với `itertools.accumulate`

Đổi mỗi ký tự thành `+1` (mở), `-1` (đóng), `0` (khác). Độ sâu sau mỗi ký tự = **tổng tiền tố** của dãy đó. `accumulate(seq)` trả về các tổng cộng dồn: `[a0, a0+a1, a0+a1+a2, ...]`.

Thêm `initial=0` để có cả giá trị ban đầu `0` (tránh lỗi `max()` của dãy rỗng — dù đề có `len >= 1`, chuỗi `"1"` vẫn cho độ sâu 0).

In [11]:
from itertools import accumulate
print(list(accumulate([1, 1, -1, 1, -1, -1])))
print(list(accumulate([1, -1], initial=0)))

def max_depth_accumulate(s: str) -> int:
    steps = [1 if ch == "(" else -1 if ch == ")" else 0 for ch in s]
    return max(accumulate(steps, initial=0))

print([max_depth_accumulate(s) for s, _ in examples], max_depth_accumulate("1"))

[1, 2, 1, 2, 1, 0]
[0, 1, 0]
[3, 3, 3] 0


Biểu thức `1 if ch == "(" else -1 if ch == ")" else 0` là **ternary lồng nhau**: đọc từ trái sang phải — "nếu mở thì 1, ngược lại nếu đóng thì −1, ngược lại 0".

In [12]:
for ch in "(x)":
    print(repr(ch), "->", 1 if ch == "(" else -1 if ch == ")" else 0)

'(' -> 1
'x' -> 0
')' -> -1


## 8. TRACE từng bước

Trace `s = "(1+(2*3)+((8)/4))+1"` (chỉ ghi các ký tự ngoặc, ký tự khác không đổi `depth`):

| i | ch | depth sau | best sau | Ghi chú |
|:-:|:-:|:-:|:-:|---|
| 0 | `(` | 1 | 1 | vào lớp 1 |
| 3 | `(` | 2 | 2 | `(2*3)` |
| 7 | `)` | 1 | 2 | ra |
| 9 | `(` | 2 | 2 | |
| 10 | `(` | 3 | **3** | chứa `8` |
| 12 | `)` | 2 | 3 | |
| 15 | `)` | 1 | 3 | |
| 16 | `)` | 0 | 3 | về mặt đất |

Kết quả `3`.

In [13]:
def trace_max_depth(s: str) -> int:
    depth = best = 0
    for i, ch in enumerate(s):
        if ch == "(":
            depth += 1
            note = "mở"
            if depth > best:
                best = depth
                note += " -> kỷ lục mới!"
        elif ch == ")":
            depth -= 1
            note = "đóng"
        else:
            continue                      # bỏ qua ký tự không phải ngoặc khi in
        print(f"i={i:>2} ch={ch} depth={depth} best={best} | {'  ' * depth}{note}")
    return best

print("Kết quả:", trace_max_depth("(1+(2*3)+((8)/4))+1"))

i= 0 ch=( depth=1 best=1 |   mở -> kỷ lục mới!
i= 3 ch=( depth=2 best=2 |     mở -> kỷ lục mới!
i= 7 ch=) depth=1 best=2 |   đóng
i= 9 ch=( depth=2 best=2 |     mở
i=10 ch=( depth=3 best=3 |       mở -> kỷ lục mới!
i=12 ch=) depth=2 best=3 |     đóng
i=15 ch=) depth=1 best=3 |   đóng
i=16 ch=) depth=0 best=3 | đóng
Kết quả: 3


In [14]:
print("Kết quả:", trace_max_depth("()(())((()()))"))

i= 0 ch=( depth=1 best=1 |   mở -> kỷ lục mới!
i= 1 ch=) depth=0 best=1 | đóng
i= 2 ch=( depth=1 best=1 |   mở
i= 3 ch=( depth=2 best=2 |     mở -> kỷ lục mới!
i= 4 ch=) depth=1 best=2 |   đóng
i= 5 ch=) depth=0 best=2 | đóng
i= 6 ch=( depth=1 best=2 |   mở
i= 7 ch=( depth=2 best=2 |     mở
i= 8 ch=( depth=3 best=3 |       mở -> kỷ lục mới!
i= 9 ch=) depth=2 best=3 |     đóng
i=10 ch=( depth=3 best=3 |       mở
i=11 ch=) depth=2 best=3 |     đóng
i=12 ch=) depth=1 best=3 |   đóng
i=13 ch=) depth=0 best=3 | đóng
Kết quả: 3


Cột thụt lề `'  ' * depth` giúp **nhìn thấy** độ sâu như cây thư mục. Ví dụ 3 cho thấy kỷ lục 3 chỉ đạt được ở cụm cuối.

## 9. Code nộp LeetCode

### 9.1. Cách chính — biến đếm

In [15]:
class Solution:
    def maxDepth(self, s: str) -> int:
        depth = best = 0
        for ch in s:
            if ch == "(":
                depth += 1
                best = max(best, depth)
            elif ch == ")":
                depth -= 1
        return best

print([Solution().maxDepth(s) for s, _ in examples])

[3, 3, 3]


### 9.2. Các cách khác

In [16]:
class SolutionStack:
    def maxDepth(self, s: str) -> int:
        stack, best = [], 0
        for ch in s:
            if ch == "(":
                stack.append(ch)
                best = max(best, len(stack))
            elif ch == ")":
                stack.pop()
        return best

class SolutionAccumulate:
    def maxDepth(self, s: str) -> int:
        return max(accumulate((ch == "(") - (ch == ")") for ch in s), default=0)

class SolutionBrute:
    def maxDepth(self, s: str) -> int:
        return max(s[:i].count("(") - s[:i].count(")") for i in range(len(s) + 1))

for cls in [SolutionStack, SolutionAccumulate, SolutionBrute]:
    print(cls.__name__, [cls().maxDepth(s) for s, _ in examples], cls().maxDepth("7"))

SolutionStack [3, 3, 3] 0
SolutionAccumulate [3, 3, 3] 0
SolutionBrute [3, 3, 3] 0


Hai mẹo trong `SolutionAccumulate`:
- `(ch == "(") - (ch == ")")`: `True - False = 1`, `False - True = -1`, `False - False = 0` → gọn thay cho ternary.
- `max(..., default=0)`: nếu dãy rỗng thì trả `0` thay vì báo lỗi (phòng hờ; đề có `len(s) >= 1`). Không cần `initial=0` vì với VPS mọi tổng tiền tố đều `>= 0`, và tổng cuối cùng luôn là `0`, nên max không bao giờ nhỏ hơn đáp án đúng.

In [17]:
print(True - False, False - True, False - False)
print(max([], default=0))

1 -1 0
0


## 10. Vì sao đúng? Độ phức tạp

**Invariant:** sau khi đọc `k` ký tự đầu, `depth` = (số `(`) − (số `)`) trong `s[:k]` = số ngoặc mở **đang chờ đóng** = độ sâu tại vị trí đó (VPS đảm bảo `depth >= 0`).

`best` = max của `depth` qua mọi vị trí đã đọc. Độ sâu chỉ **tăng** tại dấu `(`, nên cập nhật ngay sau `(` là đủ để bắt mọi đỉnh. Đọc hết chuỗi → `best` là max toàn cục = độ sâu lồng nhau.

Gọi `n = len(s)`.

| Cách | Time | Space phụ |
|---|---|---|
| 1. Đếm trước mỗi vị trí | `O(n²)` | `O(n)` (slice tạm) |
| 2. Stack | `O(n)` | `O(n)` |
| 3. Biến đếm | `O(n)` | **`O(1)`** |
| 4. `accumulate` | `O(n)` | `O(n)` (list bước) hoặc `O(1)` nếu dùng generator |

## 11. Bộ test

1. Ba ví dụ đề.
2. Không có ngoặc: `"1"`, `"1+2*3"` → 0.
3. Một cặp `"()"` → 1; lồng sâu `"((((()))))"` → 5.
4. Nhiều cụm rời nhau, cụm sâu nhất ở đầu/giữa/cuối.
5. Đối chiếu 8 hàm trên VPS sinh ngẫu nhiên.

In [18]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<26} PASS ({actual})")

solvers = [max_depth_brute, max_depth_stack, max_depth_counter, max_depth_accumulate,
           Solution().maxDepth, SolutionStack().maxDepth, SolutionAccumulate().maxDepth, SolutionBrute().maxDepth]
cases = [
    ("ví dụ 1", "(1+(2*3)+((8)/4))+1", 3),
    ("ví dụ 2", "(1)+((2))+(((3)))", 3),
    ("ví dụ 3", "()(())((()()))", 3),
    ("không ngoặc", "1", 0),
    ("biểu thức phẳng", "1+2*3", 0),
    ("một cặp", "()", 1),
    ("lồng 5", "((((()))))", 5),
    ("sâu nhất ở đầu", "((1))+(2)", 2),
    ("sâu nhất ở giữa", "(1)+((((2))))+(3)", 4),
    ("liền nhau", "()()()()", 1),
]
for name, s, expected in cases:
    for f in solvers:
        assert f(s) == expected, (name, f)
    check(name, max_depth_counter(s), expected)

ví dụ 1                    PASS (3)
ví dụ 2                    PASS (3)
ví dụ 3                    PASS (3)
không ngoặc                PASS (0)
biểu thức phẳng            PASS (0)
một cặp                    PASS (1)
lồng 5                     PASS (5)
sâu nhất ở đầu             PASS (2)
sâu nhất ở giữa            PASS (4)
liền nhau                  PASS (1)


In [19]:
import random

def random_vps(rng, depth=0, budget=None):
    # Sinh VPS ngẫu nhiên theo định nghĩa đệ quy: nối nhiều khối, mỗi khối là chữ số hoặc (VPS)
    if budget is None:
        budget = [rng.randint(1, 30)]
    parts = []
    while budget[0] > 0 and rng.random() < 0.8:
        budget[0] -= 1
        if rng.random() < 0.5 and depth < 8:
            parts.append("(" + random_vps(rng, depth + 1, budget) + ")")
        else:
            parts.append(rng.choice("0123456789+-*/"))
    return "".join(parts)

rng = random.Random(1614)
tested = 0
for _ in range(600):
    s = random_vps(rng)
    if not s:
        continue
    expected = max_depth_brute(s)
    for f in solvers:
        assert f(s) == expected
    tested += 1
print(f"{tested} VPS ngẫu nhiên: 8 hàm khớp nhau")

477 VPS ngẫu nhiên: 8 hàm khớp nhau


Hàm `random_vps` gọi **chính nó** (đệ quy) để sinh `(A)` với `A` là VPS — đúng theo định nghĩa ở mục 1.1. `budget` là list một phần tử để mọi tầng đệ quy **dùng chung** một ngân sách (list là đối tượng có thể thay đổi — mutable).

## 12. Lỗi người mới hay gặp

### Lỗi 1: Trả về `depth` cuối thay vì `best`

Với VPS, `depth` cuối **luôn bằng 0**!

In [20]:
def wrong_return_depth(s):
    depth = 0
    for ch in s:
        if ch == "(":
            depth += 1
        elif ch == ")":
            depth -= 1
    return depth

print("sai:", wrong_return_depth("(())"), "| đúng:", max_depth_counter("(())"))

sai: 0 | đúng: 2


### Lỗi 2: Đếm tổng số `(` thay vì độ sâu

In [21]:
def wrong_count_open(s):
    return s.count("(")

print("sai:", wrong_count_open("()()()"), "| đúng:", max_depth_counter("()()()"))

sai: 3 | đúng: 1


### Lỗi 3: Cập nhật `best` **trước** khi tăng `depth`

Thứ tự `best = max(best, depth); depth += 1` sẽ luôn chậm một nhịp → thiếu 1.

In [22]:
def wrong_update_order(s):
    depth = best = 0
    for ch in s:
        if ch == "(":
            best = max(best, depth)   # SAI: cập nhật trước
            depth += 1
        elif ch == ")":
            depth -= 1
    return best

print("sai:", wrong_update_order("((()))"), "| đúng:", max_depth_counter("((()))"))

sai: 2 | đúng: 3


### Lỗi 4: Coi mọi ký tự không phải `(` là `)`

`if ch == "(": ... else: depth -= 1` → chữ số và phép toán cũng làm giảm độ sâu.

In [23]:
def wrong_else(s):
    depth = best = 0
    for ch in s:
        if ch == "(":
            depth += 1
            best = max(best, depth)
        else:
            depth -= 1
    return best

s = "(1+(2*3)+((8)/4))+1"
print("sai:", wrong_else(s), "| đúng:", max_depth_counter(s))

sai: 1 | đúng: 3


## 13. Mẹo nhận dạng pattern

- Bài về ngoặc **một loại** chỉ cần **biến đếm** (`balance`); nhiều loại ngoặc (`()[]{}`) mới cần **stack** để nhớ loại.
- "Độ sâu", "mức lồng", "số lớp" → `balance` + `max`.
- Dãy `+1/-1` → **tổng tiền tố**; đỉnh của tổng tiền tố là độ sâu lớn nhất.
- Bài liên quan: 20 Valid Parentheses, 921 Minimum Add to Make Parentheses Valid, 1021 Remove Outermost Parentheses, 1111 Maximum Nesting Depth of Two Valid Parentheses Strings, 856 Score of Parentheses.

## 14. Biến thể

### 14.1. Kiểm tra chuỗi có phải VPS không (một loại ngoặc)

In [24]:
def is_vps(s: str) -> bool:
    balance = 0
    for ch in s:
        if ch == "(":
            balance += 1
        elif ch == ")":
            balance -= 1
            if balance < 0:          # đóng nhiều hơn mở -> hỏng ngay
                return False
    return balance == 0

print(is_vps("(()())"), is_vps("())("), is_vps("(("))

True False False


### 14.2. Nhiều loại ngoặc — độ sâu lớn nhất (giả sử hợp lệ), dùng stack để kiểm tra khớp loại

In [25]:
PAIRS = {")": "(", "]": "[", "}": "{"}

def max_depth_multi(s: str) -> int:
    stack, best = [], 0
    for ch in s:
        if ch in "([{":
            stack.append(ch)
            best = max(best, len(stack))
        elif ch in PAIRS:
            if not stack or stack[-1] != PAIRS[ch]:
                return -1             # không hợp lệ
            stack.pop()
    return best if not stack else -1

print(max_depth_multi("{[()]}()"), max_depth_multi("([)]"), max_depth_multi("[{(x)}]"))

3 -1 3


### 14.3. Độ sâu của từng chữ số

In ra mỗi chữ số nằm ở tầng mấy — dùng lại `depth` tại thời điểm gặp chữ số.

In [26]:
def digit_depths(s: str) -> dict:
    depth, out = 0, {}
    for ch in s:
        if ch == "(":
            depth += 1
        elif ch == ")":
            depth -= 1
        elif ch.isdigit():
            out[ch] = depth
    return out

print(digit_depths("(1+(2*3)+((8)/4))+1"))

{'1': 0, '2': 2, '3': 2, '8': 3, '4': 2}


`ch.isdigit()` trả `True` nếu ký tự là chữ số. Chú ý dict ghi đè key trùng — chữ số `1` xuất hiện 2 lần nên giữ giá trị **sau cùng** (tầng 0).

## 15. Bài tập tự luyện (TODO)

**Bài 1 — tính tay.** Viết dãy `depth` sau mỗi ký tự của `"(()(()))"` và cho biết độ sâu lớn nhất.

**Bài 2 — đoán output.** `list(accumulate([1, 1, -1, 1]))` in ra gì? `max([], default=-1)`?

### Bài 3 — Điền TODO: lời giải biến đếm

In [27]:
def exercise_max_depth(s: str) -> int:
    # TODO 1: depth = best = 0
    # TODO 2: '(' -> tăng depth rồi cập nhật best; ')' -> giảm depth; ký tự khác -> bỏ qua
    # TODO 3: return best
    pass

# assert exercise_max_depth("(1)+((2))+(((3)))") == 3

### Bài 4 — Tự viết hàm kiểm tra VPS một loại ngoặc

In [28]:
def exercise_is_vps(s: str) -> bool:
    # TODO: balance không bao giờ âm và cuối cùng bằng 0
    pass

# assert exercise_is_vps("(()())") is True
# assert exercise_is_vps("())(") is False

### Bài 5 — Biến thể: số ngoặc tối thiểu cần thêm để hợp lệ (LeetCode 921)

Chuỗi chỉ gồm `(` và `)`, có thể không hợp lệ. Đếm số ngoặc ít nhất cần chèn để thành VPS.

In [29]:
def exercise_min_add(s: str) -> int:
    # TODO: khi balance sắp âm, phải thêm một '(' ; cuối cùng thêm balance dấu ')'
    pass

# assert exercise_min_add("())") == 1
# assert exercise_min_add("(((") == 3
# assert exercise_min_add("()))((") == 4

### Bài 6 — Mini challenge: tách VPS thành 2 VPS với độ sâu lớn nhất nhỏ nhất (LeetCode 1111)

Cho VPS `seq` chỉ gồm ngoặc. Gán mỗi ký tự cho nhóm `0` hoặc `1` sao cho mỗi nhóm là VPS và `max(depth(A), depth(B))` nhỏ nhất. Trả về list nhãn.

In [30]:
def exercise_split(seq: str) -> list[int]:
    # TODO: gợi ý - nhãn theo tính chẵn lẻ của độ sâu
    pass

# Kiểm tra: hai nhóm đều VPS và độ sâu mỗi nhóm <= ceil(max_depth / 2)

## 16. Gợi ý 3 mức

**Bài 1:** M1: `(` +1, `)` −1. M2: bắt đầu `1, 2, 1, 2, 3, ...`. M3: đỉnh là 3.

**Bài 2:** M1: tổng cộng dồn. M2: `1, 2, 1, 2`. M3: `default` dùng khi dãy rỗng.

**Bài 3:** M1: xem mục 6. M2: cập nhật `best` **sau** `depth += 1`. M3: `elif ch == ")": depth -= 1`.

**Bài 4:** M1: một biến `balance`. M2: âm là hỏng ngay. M3: `return balance == 0`.

**Bài 5:** M1: hai biến `balance` và `add`. M2: gặp `)` khi `balance == 0` → `add += 1`, không giảm. M3: `return add + balance`.

**Bài 6:** M1: các tầng xen kẽ nhau giữa hai nhóm. M2: với `(`, tăng depth rồi gán nhãn `depth % 2`. M3: với `)`, gán nhãn `depth % 2` rồi mới giảm.

## 17. Đáp án tham khảo

**Bài 1:** `1, 2, 1, 2, 3, 2, 1, 0` → độ sâu lớn nhất `3`.

**Bài 2:** `[1, 2, 1, 2]` và `-1`.

In [31]:
def answer_max_depth(s):
    depth = best = 0
    for ch in s:
        if ch == "(":
            depth += 1
            best = max(best, depth)
        elif ch == ")":
            depth -= 1
    return best

def answer_is_vps(s):
    balance = 0
    for ch in s:
        if ch == "(":
            balance += 1
        elif ch == ")":
            balance -= 1
            if balance < 0:
                return False
    return balance == 0

def answer_min_add(s):
    balance = add = 0
    for ch in s:
        if ch == "(":
            balance += 1
        elif balance == 0:
            add += 1
        else:
            balance -= 1
    return add + balance

def answer_split(seq):
    depth, labels = 0, []
    for ch in seq:
        if ch == "(":
            depth += 1
            labels.append(depth % 2)
        else:
            labels.append(depth % 2)
            depth -= 1
    return labels

depths = list(accumulate(1 if c == "(" else -1 for c in "(()(()))"))
assert depths == [1, 2, 1, 2, 3, 2, 1, 0] and max(depths) == 3
assert list(accumulate([1, 1, -1, 1])) == [1, 2, 1, 2] and max([], default=-1) == -1
assert answer_max_depth("(1)+((2))+(((3)))") == 3
assert answer_is_vps("(()())") is True and answer_is_vps("())(") is False
assert answer_min_add("())") == 1 and answer_min_add("(((") == 3 and answer_min_add("()))((") == 4
for _ in range(300):
    seq = "".join(ch for ch in random_vps(rng) if ch in "()")
    if not seq:
        continue
    labels = answer_split(seq)
    A = "".join(c for c, l in zip(seq, labels) if l == 0)
    Bs = "".join(c for c, l in zip(seq, labels) if l == 1)
    assert answer_is_vps(A) and answer_is_vps(Bs)
    assert max(answer_max_depth(A), answer_max_depth(Bs)) == (answer_max_depth(seq) + 1) // 2
    assert answer_is_vps(seq) and answer_min_add(seq) == 0
print("Đáp án tham khảo: PASS toàn bộ!")

Đáp án tham khảo: PASS toàn bộ!


## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def maxDepth(self, s: str) -> int:
        depth = best = 0
        for ch in s:
            if ch == "(":
                depth += 1
                best = max(best, depth)   # cập nhật SAU khi tăng
            elif ch == ")":
                depth -= 1
        return best                       # không phải depth (luôn = 0)
```

**Câu thần chú:** *"Mở +1, đóng −1, ký tự khác bỏ qua — nhớ đỉnh cao nhất."*

**Checklist trước khi submit:**
- [ ] Trả `best`, không trả `depth`?
- [ ] Cập nhật `best` **sau** `depth += 1`?
- [ ] Dùng `elif ch == ")"` (không dùng `else` cho mọi ký tự khác)?
- [ ] Chuỗi không có ngoặc trả `0`?
- [ ] Tên method đúng `maxDepth`?